# 04 – Unsupervised Learning & Advanced Preprocessing (Preetham's Scikit-learn Mastery Series)

Welcome to **Part 4** of the Scikit-Learn Mastery Series! In supervised learning, models are guided by target labels $y$. In **unsupervised learning**, we uncover hidden patterns, structures, groupings, and representations directly from feature matrix $X$ without explicit labels.

---

### 📌 What You Will Master in This Notebook:
1. **Clustering Algorithms**: K-Means, Agglomerative Hierarchical Clustering, and DBSCAN.
2. **Clustering Evaluation**: Elbow Method, Silhouette Analysis, ARI, NMI, Calinski-Harabasz, and Davies-Bouldin metrics.
3. **Dimensionality Reduction**: Principal Component Analysis (PCA), TruncatedSVD, and t-SNE manifold learning.
4. **Feature Selection Strategies**: Filter methods (`SelectKBest`), Wrapper methods (`RFE`), and Embedded L1 methods (`SelectFromModel`).
5. **Missing Value Imputation**: `SimpleImputer`, `KNNImputer`, and `IterativeImputer` (MICE).
6. **Anomaly & Outlier Detection**: `IsolationForest` and `LocalOutlierFactor` (LOF).
7. **End-to-End Unsupervised Pipeline**: Integrating preprocessing, reduction, and clustering seamlessly.


In [2]:
# Setup, reproducible seeds, and dataset preparation
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn import datasets
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

# Set aesthetic plot styling
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['font.size'] = 11

# Load Wine dataset (13 continuous features, 3 class labels for evaluation)
wine = datasets.load_wine()
X, y = wine.data, wine.target
feature_names = wine.feature_names
target_names = wine.target_names

# Standardize features (Mean = 0, Variance = 1) - CRITICAL for distance-based & variance-based methods
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("Dataset Loaded: Wine Dataset")
print(f"Samples (n): {X.shape[0]} | Features (d): {X.shape[1]}")
print(f"Target Classes: {target_names}")
print(f"Scaled Mean (first 3): {X_scaled.mean(axis=0)[:3].round(4)}")
print(f"Scaled Std  (first 3): {X_scaled.std(axis=0)[:3].round(4)}")


Dataset Loaded: Wine Dataset
Samples (n): 178 | Features (d): 13
Target Classes: ['class_0' 'class_1' 'class_2']
Scaled Mean (first 3): [ 0.  0. -0.]
Scaled Std  (first 3): [1. 1. 1.]


---
## 1. K-Means Clustering

### 📐 Mathematical Formulation
K-Means partitions dataset $X = \{x_1, x_2, \dots, x_n\}$ into $k$ distinct, non-overlapping clusters $C = \{C_1, C_2, \dots, C_k\}$ by minimizing the **Within-Cluster Sum of Squares (WCSS)**, also known as **Inertia**:

$$J = \sum_{j=1}^{k} \sum_{x_i \in C_j} \|x_i - \mu_j\|^2$$

where $\mu_j$ is the centroid (mean vector) of cluster $C_j$:

$$\mu_j = \frac{1}{|C_j|} \sum_{x_i \in C_j} x_i$$

### 🔄 Algorithm Steps (Lloyd's Algorithm):
1. **Initialize**: Choose $k$ initial centroids $\mu_1, \dots, \mu_k$ (e.g., using `k-means++` smart seeding).
2. **Assign**: Assign each data point $x_i$ to the nearest centroid:
   $$C_j = \{ x_i : \|x_i - \mu_j\| \le \|x_i - \mu_l\|, \forall l \in \{1, \dots, k\} \}$$
3. **Update**: Recalculate each centroid as the mean of all points assigned to that cluster.
4. **Iterate**: Repeat Steps 2 & 3 until centroids stabilize (convergence) or max iterations are reached.


In [4]:
from sklearn.cluster import KMeans

# Fit K-Means with k=3 (matching the 3 wine classes)
kmeans = KMeans(n_clusters=3, init='k-means++', n_init=10, random_state=42)
clusters_km = kmeans.fit_predict(X_scaled)

print("K-Means Results:")
print(f"Inertia (WCSS): {kmeans.inertia_:.2f}")
print(f"Number of iterations to converge: {kmeans.n_iter_}")
print(f"Cluster centroids shape: {kmeans.cluster_centers_.shape}")
print(f"Cluster sample distribution: {np.bincount(clusters_km)}")

# Compare random init vs k-means++ init
km_random = KMeans(n_clusters=3, init='random', n_init=1, random_state=42).fit(X_scaled)
km_pp = KMeans(n_clusters=3, init='k-means++', n_init=1, random_state=42).fit(X_scaled)
print(f"Inertia (1 pass random init): {km_random.inertia_:.2f}")
print(f"Inertia (1 pass k-means++): {km_pp.inertia_:.2f}")


K-Means Results:
Inertia (WCSS): 1277.93
Number of iterations to converge: 7
Cluster centroids shape: (3, 13)
Cluster sample distribution: [65 51 62]
Inertia (1 pass random init): 1282.46
Inertia (1 pass k-means++): 1277.93
